In [32]:
import pandas as pd
df = pd.read_csv("/workspaces/Loan-underwriting/data/raw/Customer_Data.csv")

print(df.head())

In [35]:
processed_df = df[df["Status"] != "Rejected"].copy()
processed_df["Default"] = (processed_df["DPD"] > 0).astype(int)
print(processed_df.head(3))

   LAN Customer ID Customer Name  Age        State         District  \
0    4         FE4  Monika Gupta   39    Karnataka           Mysuru   
1    5         FE5   Priya Wadwa   33    Karnataka  Channarayapatna   
2    6         FE6      Tarang S   59  Maharashtra             Pune   

  Manufacturer Vehicle_Model Existing Customer  Loan amount requested  \
0         KOMA       KX00003               Yes                 448588   
1         KOMA       KX00003                No                 754946   
2         KOMA       KX00003               Yes                 895696   

   Loan_tenure(years)  Cibil Score  Obligation  Land acres   Crop  \
0                   3          747       87666           2   Rice   
1                   6          677       89413           2  Maize   
2                   4          717      157834           1   Rice   

   Other_documented_income    DPD     Status  Default  
0                   498178    0.0  Disbursed        0  
1                   330556    0.0

In [36]:
#Joining Customer data with Crop Income table 
crop_income_df = pd.read_csv(
    "/workspaces/Loan-underwriting/data/raw/Crop_Income.csv"
)
print(crop_income_df.head(3))

     State   District   Crop  Productivity(quintal/acre)  Rate(Rs/quintal)
0  Gujarat  Ahmedabad  Wheat                          13              2485
1  Gujarat  Ahmedabad  Maize                          10              2358
2  Gujarat  Ahmedabad   Rice                          21              2703


In [37]:
#Joining Customer data with Model cost table 
assetcost_df = pd.read_csv(
    "/workspaces/Loan-underwriting/data/raw/Model_cost.csv"
)
print(assetcost_df.head(3))

  Manufacturer Approved Vehicle_Model  Net Dealer Price
0         MAHI      Yes       MX00001            813356
1         MAHI      Yes       MX00002           1392077
2         MAHI      Yes       MX00003            729957


In [38]:
processed_df = processed_df.merge(
    assetcost_df,
    on=["Manufacturer", "Vehicle_Model"],
    how="left"
)
print(processed_df.head(20))

    LAN Customer ID  Customer Name  Age           State         District  \
0     4         FE4   Monika Gupta   39       Karnataka           Mysuru   
1     5         FE5    Priya Wadwa   33       Karnataka  Channarayapatna   
2     6         FE6       Tarang S   59     Maharashtra             Pune   
3     9         FE9    Sita Kumari   25  Madhya Pradesh        Dhulagori   
4    10        FE10   Suraj Sharma   47         Gujarat        Ahmedabad   
5    13        FE13   Anjali Wadwa   39       Karnataka        Bengaluru   
6    16        FE16   Ankit Sharma   35      Tamil Nadu       Coimbatore   
7    17        FE17   Geeta Kumari   28         Gujarat        Ahmedabad   
8    19        FE19    Priya Wadwa   54   Uttar Pradesh          Lucknow   
9    20        FE20       Tarang S   31       Karnataka           Mysuru   
10   22        FE22    Rita Kumari   32         Gujarat        Ahmedabad   
11   23        FE23    Sita Kumari   40         Gujarat            Surat   
12   27     

In [39]:
processed_df = processed_df.merge(
    crop_income_df,
    on=["State", "District", "Crop"],
    how="left"
)
print(processed_df.head(3))

   LAN Customer ID Customer Name  Age        State         District  \
0    4         FE4  Monika Gupta   39    Karnataka           Mysuru   
1    5         FE5   Priya Wadwa   33    Karnataka  Channarayapatna   
2    6         FE6      Tarang S   59  Maharashtra             Pune   

  Manufacturer Vehicle_Model Existing Customer  Loan amount requested  ...  \
0         KOMA       KX00003               Yes                 448588  ...   
1         KOMA       KX00003                No                 754946  ...   
2         KOMA       KX00003               Yes                 895696  ...   

   Land acres   Crop  Other_documented_income    DPD     Status  Default  \
0           2   Rice                   498178    0.0  Disbursed        0   
1           2  Maize                   330556    0.0  Disbursed        0   
2           1   Rice                   273521  114.0  Disbursed        1   

   Approved Net Dealer Price  Productivity(quintal/acre) Rate(Rs/quintal)  
0        No          

In [40]:
#Creating New Features: Crop income, Total income, Annual Installment, NFCF:AI
processed_df["Crop_Income"] = (processed_df["Land acres"]*processed_df["Productivity(quintal/acre)"]*processed_df["Rate(Rs/quintal)"]*2)

processed_df["Total_Income"] = (processed_df["Crop_Income"]+processed_df["Other_documented_income"])

processed_df["LTV"] = (processed_df["Loan amount requested"]/processed_df["Net Dealer Price"])

processed_df["NFCF"] = (processed_df["Total_Income"]-processed_df["Obligation"])


def calculate_annual_installment(row):

    P = row["Loan amount requested"]

    r = 14.5 / 100

    n = row["Loan_tenure(years)"]

    return ( P * r * (1 + r) ** n) / ( (1 + r) ** n - 1 )


processed_df["Annual_Installment"] = (
   processed_df.apply(
        calculate_annual_installment,
        axis=1
    )
)


processed_df["NFCF:AI"] = (processed_df["NFCF"]/processed_df["Annual_Installment"])

print(processed_df.head(3))

   LAN Customer ID Customer Name  Age        State         District  \
0    4         FE4  Monika Gupta   39    Karnataka           Mysuru   
1    5         FE5   Priya Wadwa   33    Karnataka  Channarayapatna   
2    6         FE6      Tarang S   59  Maharashtra             Pune   

  Manufacturer Vehicle_Model Existing Customer  Loan amount requested  ...  \
0         KOMA       KX00003               Yes                 448588  ...   
1         KOMA       KX00003                No                 754946  ...   
2         KOMA       KX00003               Yes                 895696  ...   

   Approved  Net Dealer Price  Productivity(quintal/acre)  Rate(Rs/quintal)  \
0        No            788205                          21              2587   
1        No            788205                          12              2001   
2        No            788205                          20              2547   

  Crop_Income  Total_Income       LTV    NFCF  Annual_Installment   NFCF:AI  
0      

In [41]:
processed_df = processed_df.drop(columns=["DPD"])

In [42]:
#Univariate analysis
correlation_matrix = processed_df.corr( numeric_only=True)
default_correlation = (
    correlation_matrix["Default"]
    .sort_values(ascending=False)
)

print(default_correlation)

Default                       1.000000
Loan amount requested         0.139838
Annual_Installment            0.110511
LTV                           0.105392
LAN                           0.013508
Net Dealer Price              0.004533
Loan_tenure(years)           -0.003710
Productivity(quintal/acre)   -0.004342
Rate(Rs/quintal)             -0.018980
Obligation                   -0.020329
Cibil Score                  -0.028071
Other_documented_income      -0.087203
Crop_Income                  -0.118151
Land acres                   -0.132920
NFCF                         -0.135388
Total_Income                 -0.144858
Age                          -0.148498
NFCF:AI                      -0.149916
Name: Default, dtype: float64


In [43]:
processed_df["Existing Customer"] = (
    processed_df["Existing Customer"]
    .map({"Yes": 1, "No": 0}))

(
    processed_df.groupby("Existing Customer")["Default"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

Existing Customer
0    5.183051
1    2.002305
Name: Default, dtype: float64

In [44]:
(
    processed_df.groupby("District")["Default"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

District
Bishanpura         4.430380
Mysuru             4.385001
Coimbatore         4.317549
Bengaluru          4.171966
Surat              4.156389
Lucknow            4.152908
Kanpur             4.067132
Dhulagori          4.060249
Ahmedabad          3.939658
Nagpur             3.801778
Hyderabad          3.654050
Udaipur            3.595700
Manjari            3.576848
Nellikuppam        3.559624
Pune               3.495702
Channarayapatna    3.400756
Name: Default, dtype: float64

In [45]:
processed_df["LTV_BKT"]= pd.cut(
    processed_df["LTV"],
    bins=[0, 0.5, 0.7, 0.9, 1.0],
    labels=["<50%", "50-70%", "70-90%", ">90%"]
)

processed_df.groupby("LTV_BKT")["Default"].mean() * 100

LTV_BKT
<50%      0.168884
50-70%    4.301140
70-90%    8.028476
>90%      1.350522
Name: Default, dtype: float64

In [46]:
age_groups = pd.cut(
    processed_df["Age"],
    bins=[18, 30, 45, 60, 100],
    labels=["18-30", "31-45", "46-60", "60+"]
)

processed_df.groupby(age_groups)["Default"].mean() * 100

Age
18-30    10.127755
31-45     1.908240
46-60     2.034053
Name: Default, dtype: float64

In [47]:
processed_df["Land_BKT"]= pd.cut(
                   processed_df["Land acres"],
                   bins=[0,1,2,4,10,20],
                   labels=["<1","1-2", "2-4", "4-10",">10"]
)

processed_df.groupby("Land_BKT")["Default"].mean() * 100

Land_BKT
<1      6.866686
1-2     5.002672
2-4     0.243922
4-10    0.000000
Name: Default, dtype: float64

In [48]:
processed_df["Income_BKT"]= pd.cut(
                   processed_df["Total_Income"],
                   bins=[0,100000,200000,500000,800000,1000000],
                   labels=["<100000","100000-200000", "200000-500000", "500000-800000",">800000"]
)

processed_df.groupby("Income_BKT")["Default"].mean() * 100

Income_BKT
100000-200000    24.637681
200000-500000     7.805632
500000-800000     3.328321
>800000           0.156817
Name: Default, dtype: float64

In [49]:
processed_df["Loanamount_BKT"]= pd.cut(
                   processed_df["Loan amount requested"],
                   bins=[0,200000,500000,600000,800000,1000000,2000000],
                   labels=["<200000","200000-500000", "500000-600000", "600000-800000","800000-1000000",">1000000"]
)

processed_df.groupby("Loanamount_BKT")["Default"].mean() * 100

Loanamount_BKT
200000-500000     1.230897
500000-600000     0.864692
600000-800000     4.721377
800000-1000000    7.723419
Name: Default, dtype: float64

In [50]:
#Cash-flow analysis
processed_df["NFCFAI_BKT"]= pd.cut(
                   processed_df["NFCF:AI"],
                   bins=[-2,-1, 0,1,2,4,10],
                   labels=["<-1","-1-0", "0-1", "1-2","2-4",">4"]
)

processed_df.groupby("NFCFAI_BKT")["Default"].mean() * 100

NFCFAI_BKT
-1-0    14.792899
0-1     10.459305
1-2      7.509082
2-4      1.946399
>4       0.049978
Name: Default, dtype: float64

In [51]:
#agricultural capacity
#Land acres × Crop_Income → Default

crop_groups = pd.qcut(
    processed_df["Crop_Income"],
    q=4
)

default_rate_table = pd.pivot_table(
    processed_df,
    values="Default",
    index="Land_BKT",
    columns=crop_groups,
    aggfunc="mean",
    observed=True
) * 100

default_rate_table.round(2)


Crop_Income,"(8023.999, 103440.0]","(103440.0, 169728.0]","(169728.0, 320418.0]","(320418.0, 1424160.0]"
Land_BKT,,,,
<1,7.23,6.43,5.13,NaN
1-2,6.40,5.78,4.21,2.12
2-4,0.19,0.15,0.35,0.18
4-10,0.00,0.00,0.00,0.00


In [52]:
#Credit Risk

#Cibil Score × LTV → Default

CIBIL_BKT = pd.qcut(
    processed_df["Cibil Score"],
    q=4
)

default_rate_table2 = pd.pivot_table(
    processed_df,
    values="Default",
    index=CIBIL_BKT,
    columns="LTV_BKT",
    aggfunc="mean",
    observed=True
) * 100

default_rate_table2.round(2)


LTV_BKT,<50%,50-70%,70-90%,>90%
Cibil Score,,,,
"(599.999, 638.0]",0.15,4.09,10.10,1.60
"(638.0, 676.0]",0.19,4.41,10.30,1.76
"(676.0, 713.0]",0.19,4.47,7.62,1.25
"(713.0, 750.0]",0.14,4.24,3.92,0.76


In [53]:
#Credit Risk
#Loan amount × LTV → Default

default_rate_table3 = pd.pivot_table(
    processed_df,
    values="Default",
    index="Loanamount_BKT",
    columns="LTV_BKT",
    aggfunc="mean",
    observed=True
) * 100

default_rate_table3.round(2)

LTV_BKT,<50%,50-70%,70-90%,>90%
Loanamount_BKT,,,,
200000-500000,0.29,2.06,0.00,NaN
500000-600000,0.09,1.98,0.00,NaN
600000-800000,0.03,5.26,7.35,1.59
800000-1000000,NaN,7.59,10.71,1.22


In [54]:
features = [

    "Loan amount requested",

    "Existing Customer",

    "Annual_Installment",

    "LTV",

    "Land acres",

    "Total_Income",

    "Age",

    "NFCF",

    "NFCF:AI",

    "Cibil Score",

    "Obligation",

    "Loan_tenure(years)"

]

In [55]:
#Using Standardized logistic regression
%pip install statsmodels
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
X = processed_df[features]

y = processed_df["Default"]

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#logistic regression
logit_model = sm.Logit(y, X_scaled)

result = logit_model.fit()

print(result.summary())


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
         Current function value: 0.687353
         Iterations: 35


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                           Logit Regression Results                           
Dep. Variable:                Default   No. Observations:               112973
Model:                          Logit   Df Residuals:                   112962
Method:                           MLE   Df Model:                           10
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                  -3.142
Time:                        19:28:27   Log-Likelihood:                -77652.
converged:                      False   LL-Null:                       -18747.
Covariance Type:            nonrobust   LLR p-value:                     1.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.2191      0.028      7.841      0.000       0.164       0.274
x2            -0.0612      0.006    -10.214      0.000      -0.073      -0.049
x3            -0.0603      0.031     -1.957      0.0

In [56]:
results_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": result.params.values,
    "P_Value": result.pvalues.values,
    "CI_Lower": result.conf_int()[0].values,
    "CI_Upper": result.conf_int()[1].values
})

results_table["Significant_5%"] = results_table["P_Value"] < 0.05
results_table.round(2)

,Feature,Coefficient,P_Value,CI_Lower,CI_Upper,Significant_5%
0,Loan amount requested,0.22,0.00,0.16,0.27,True
1,Existing Customer,-0.06,0.00,-0.07,-0.05,True
2,Annual_Installment,-0.06,0.05,-0.12,0.00,False
3,LTV,0.02,0.00,0.01,0.04,True
4,Land acres,-0.05,0.00,-0.06,-0.03,True
5,Total_Income,-0.12,1.00,-333718.93,333718.69,False
6,Age,-0.11,0.00,-0.13,-0.10,True
7,NFCF,-0.11,1.00,-342848.22,342848.00,False
8,NFCF:AI,0.19,0.00,0.15,0.23,True
9,Cibil Score,-0.02,0.00,-0.03,-0.01,True


In [57]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = processed_df[features].copy()

X_vif = X_vif.fillna(X_vif.median())

X_vif = sm.add_constant(X_vif)

vif = pd.DataFrame()

vif["Feature"] = X_vif.columns

vif["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

print(vif)

                  Feature         VIF
0                   const  447.955482
1   Loan amount requested   21.706516
2       Existing Customer    1.002717
3      Annual_Installment   26.406929
4                     LTV    1.689936
5              Land acres    2.222166
6            Total_Income         inf
7                     Age    1.002458
8                    NFCF         inf
9                 NFCF:AI   12.795025
10            Cibil Score    1.000189
11             Obligation         inf
12     Loan_tenure(years)   12.714988


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [66]:
features_clean = [
    "NFCF",
    "Loan amount requested",
    "LTV",
    "Land acres",
    "Cibil Score",
    "Age",
    "Existing Customer",
    "Loan_tenure(years)"
]

M = processed_df[features_clean]

n = processed_df["Default"]

# Standardize
scaler = StandardScaler()
M_scaled = scaler.fit_transform(M)

#logistic regression
logit_model = sm.Logit(n, M_scaled)

result = logit_model.fit()

print(result.summary())

results_table_final= pd.DataFrame({
    "features_clean": M.columns,
    "Coefficient": result.params.values,
    "P_Value": result.pvalues.values,
    "CI_Lower": result.conf_int()[0].values,
    "CI_Upper": result.conf_int()[1].values
})

results_table_final["Significant_5%"] = results_table_final["P_Value"] < 0.05
results_table_final.round(2)

Optimization terminated successfully.
         Current function value: 0.687819
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                Default   No. Observations:               112973
Model:                          Logit   Df Residuals:                   112965
Method:                           MLE   Df Model:                            7
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                  -3.145
Time:                        19:40:21   Log-Likelihood:                -77705.
converged:                       True   LL-Null:                       -18747.
Covariance Type:            nonrobust   LLR p-value:                     1.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1            -0.0651      0.009     -7.638      0.000      -0.082      -0.048
x2             0.0966      0.

,features_clean,Coefficient,P_Value,CI_Lower,CI_Upper,Significant_5%
0,NFCF,-0.07,0.00,-0.08,-0.05,True
1,Loan amount requested,0.10,0.00,0.08,0.11,True
2,LTV,0.02,0.01,0.01,0.04,True
3,Land acres,-0.06,0.00,-0.08,-0.04,True
4,Cibil Score,-0.02,0.00,-0.03,-0.01,True
5,Age,-0.12,0.00,-0.13,-0.10,True
6,Existing Customer,-0.06,0.00,-0.07,-0.05,True
7,Loan_tenure(years),-0.00,0.64,-0.01,0.01,False
